# Public Full-NZ Power BI Deck.GL Demo Data

This notebook builds public demonstration data for the Power BI Deck.GL map visual. It covers five visual geometry examples:

1. LINZ places as `polygon` rows
2. LINZ road centrelines as `path` rows
3. Stats NZ Census 2023 travel-to-work OD flows as `arc` rows
4. NZTA state highway traffic count sites as `point` rows
5. Auckland Transport and Metlink GTFS ferry routes as straight A-to-B `line` rows

The notebook is offline-first. By default it reads prepared full-NZ artifacts from `PBI_DECKGL_ARTIFACT_DIR` and writes visual-ready CSVs to `PBI_DECKGL_OUTPUT_DIR`. Set `REFRESH_FROM_SOURCE=true` only when you want to rebuild the prepared LINZ and Stats NZ artifacts from source APIs. Public no-key demo sources can be refreshed independently with `REFRESH_PUBLIC_DEMO_SOURCES=true`.


## Configuration

Public users should normally download prepared artifacts, set `PBI_DECKGL_ARTIFACT_DIR` to that folder, and run the notebook without API keys. Source refresh for LINZ and Stats NZ is optional and requires `LINZ_API_KEY` or `LDS_API_KEY`, plus `DATAFINDER_API_KEY`. Public no-key refresh downloads NZTA traffic count sites, the Hamilton TLA boundary, Hamilton SA2 polygons with more than 90% of their area inside the TLA, plus Auckland Transport and Metlink GTFS ferry feeds.


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
import wkp
from branca.colormap import linear


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "environment.yml").exists() and (candidate / "notebooks").exists():
            return candidate
    return start


def env_flag(name: str, default: bool = False) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


def env_path(name: str, default: str) -> Path:
    value = os.getenv(name, default)
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path.resolve()


PROJECT_ROOT = find_project_root()
POWER_BI_NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "analysis" / "power_bi"
if str(POWER_BI_NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(POWER_BI_NOTEBOOK_DIR))

from public_demo_geometry import (
    OPTIONAL_PUBLIC_ARTIFACT_FILES,
    PUBLIC_ARTIFACT_FILES,
    PUBLIC_SOURCE_LAYERS,
    add_visual_tooltips,
    build_hamilton_tla_demo_tables,
    build_multigeometry_road_density_map,
    load_public_artifacts,
    prepare_ferry_route_line,
    prepare_traffic_count_site_point,
    public_artifacts_available,
    refresh_public_source_artifacts,
    validate_ferry_line_coordinates,
    validate_traffic_point_coordinates,
    visual_field_mapping_records,
)

ARTIFACT_DIR = env_path("PBI_DECKGL_ARTIFACT_DIR", "data/power_bi")
POWER_BI_OUTPUT_DIR = env_path("PBI_DECKGL_OUTPUT_DIR", "data/power_bi")
HAMILTON_DEMO_OUTPUT_DIR = POWER_BI_OUTPUT_DIR / "hamilton_tla_demo"
REFRESH_FROM_SOURCE = env_flag("REFRESH_FROM_SOURCE", False)
REFRESH_PUBLIC_DEMO_SOURCES = env_flag("REFRESH_PUBLIC_DEMO_SOURCES", REFRESH_FROM_SOURCE)
EXPORT_WKT_DEBUG = env_flag("EXPORT_WKT_DEBUG", False)
BUILD_PREVIEW_MAP = env_flag("BUILD_PREVIEW_MAP", False)

ANALYSIS_CRS = "EPSG:2193"
OUTPUT_CRS = "EPSG:4326"
WKP_PRECISION = int(os.getenv("WKP_PRECISION", "6"))
PLACE_SIMPLIFY_TOLERANCE_M = float(os.getenv("PLACE_SIMPLIFY_TOLERANCE_M", "25"))
ROAD_SIMPLIFY_TOLERANCE_M = float(os.getenv("ROAD_SIMPLIFY_TOLERANCE_M", "10"))
POWER_BI_ROW_WINDOW = int(os.getenv("POWER_BI_ROW_WINDOW", "10000"))

LINZ_ROAD_CENTRELINE_LAYER_ID = 50329
LINZ_SUBURB_LAYER_ID = 113764
DATAFINDER_SA2_LAYER_ID = 111227
LINZ_WFS_URL_TEMPLATE = "https://data.linz.govt.nz/services;key={api_key}/wfs"
DATAFINDER_WFS_URL_TEMPLATE = "https://datafinder.stats.govt.nz/services;key={api_key}/wfs"
DATAFINDER_OD_CSV_URL = "https://www.arcgis.com/sharing/rest/content/items/fedc12523d4f4da08f094cf13bb21807/data"
LINZ_PLACE_TYPES_FOR_MODEL = ("Conservation Land", "Island", "Suburb", "Locality")

RAW_ARTIFACT_FILES = {
    "places": "linz_places_nztm.parquet",
    "roads": "linz_road_centrelines_nztm.parquet",
    "sa2": "statsnz_sa2_2023_generalised_nztm.parquet",
    "od": "statsnz_travel_to_work_od_2023.csv",
}

LEGACY_EXPORT_FILES = {
    "places": "nz_place_dimension.csv",
    "roads": "nz_road_dimension.csv",
    "place_surface_fact": "nz_place_surface_density_fact.csv",
    "place_road_bridge": "nz_place_road_bridge.csv",
    "road_surface_dimension": "nz_road_surface_dimension.csv",
    "sa2_reference": "nz_sa2_reference.csv",
    "od_arc": "nz_sa2_travel_to_work_od_2023.csv",
}

SOURCE_LAYERS = {
    "linz_road_centrelines": {
        "layer_id": LINZ_ROAD_CENTRELINE_LAYER_ID,
        "url": "https://data.linz.govt.nz/layer/50329-nz-road-centrelines-topo-150k/",
    },
    "linz_suburbs_and_localities": {
        "layer_id": LINZ_SUBURB_LAYER_ID,
        "url": "https://data.linz.govt.nz/layer/113764-nz-suburbs-and-localities/",
    },
    "statsnz_sa2_2023_generalised": {
        "layer_id": DATAFINDER_SA2_LAYER_ID,
        "url": "https://datafinder.stats.govt.nz/layer/111227-statistical-area-2-2023-generalised/",
    },
    "statsnz_travel_to_work_od_2023": {"url": DATAFINDER_OD_CSV_URL},
}
SOURCE_LAYERS.update(PUBLIC_SOURCE_LAYERS)

POWER_BI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
wkp_context = wkp.Context()

print(f"Project root: {PROJECT_ROOT}")
print(f"Artifact directory: {ARTIFACT_DIR}")
print(f"Output directory: {POWER_BI_OUTPUT_DIR}")
print(f"Hamilton demo output directory: {HAMILTON_DEMO_OUTPUT_DIR}")
print(f"Refresh from source: {REFRESH_FROM_SOURCE}")
print(f"Refresh public demo sources: {REFRESH_PUBLIC_DEMO_SOURCES}")
print(f"Export WKT debug columns: {EXPORT_WKT_DEBUG}")


## Helper Functions

These helpers keep public behavior explicit: API keys come only from environment variables, cached artifacts are preferred, geometry is converted to EPSG:4326 for the visual, and exports are validated before the manifest is written.


In [ ]:
def get_env_api_key(*names: str) -> str:
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    joined = " or ".join(names)
    raise RuntimeError(f"Set {joined}, or run with REFRESH_FROM_SOURCE=false and prepared artifacts.")


def fetch_wfs_layer(
    *,
    service_url_template: str,
    api_key: str,
    layer_id: int,
    page_size: int = 50000,
    srs_name: str = ANALYSIS_CRS,
    timeout_seconds: int = 300,
) -> gpd.GeoDataFrame:
    features = []
    start_index = 0
    while True:
        response = requests.get(
            service_url_template.format(api_key=api_key),
            params={
                "service": "WFS",
                "version": "2.0.0",
                "request": "GetFeature",
                "typeNames": f"layer-{layer_id}",
                "outputFormat": "json",
                "count": page_size,
                "startIndex": start_index,
                "srsName": srs_name,
            },
            timeout=timeout_seconds,
        )
        response.raise_for_status()
        payload = response.json()
        page_features = payload.get("features", [])
        returned = payload.get("numberReturned", len(page_features))
        print(f"Layer {layer_id}: fetched {returned:,} features starting at {start_index:,}.")
        features.extend(page_features)
        if returned < page_size:
            break
        start_index += page_size

    return gpd.GeoDataFrame.from_features(features, crs=srs_name)


def refresh_source_artifacts(artifact_dir: Path) -> dict[str, object]:
    artifact_dir.mkdir(parents=True, exist_ok=True)
    linz_key = get_env_api_key("LINZ_API_KEY", "LDS_API_KEY")
    datafinder_key = get_env_api_key("DATAFINDER_API_KEY")

    places = fetch_wfs_layer(
        service_url_template=LINZ_WFS_URL_TEMPLATE,
        api_key=linz_key,
        layer_id=LINZ_SUBURB_LAYER_ID,
        page_size=10000,
    )
    roads = fetch_wfs_layer(
        service_url_template=LINZ_WFS_URL_TEMPLATE,
        api_key=linz_key,
        layer_id=LINZ_ROAD_CENTRELINE_LAYER_ID,
        page_size=50000,
    )
    sa2 = fetch_wfs_layer(
        service_url_template=DATAFINDER_WFS_URL_TEMPLATE,
        api_key=datafinder_key,
        layer_id=DATAFINDER_SA2_LAYER_ID,
        page_size=10000,
    )
    od = pd.read_csv(DATAFINDER_OD_CSV_URL, low_memory=False)

    places.to_parquet(artifact_dir / RAW_ARTIFACT_FILES["places"], index=False)
    roads.to_parquet(artifact_dir / RAW_ARTIFACT_FILES["roads"], index=False)
    sa2.to_parquet(artifact_dir / RAW_ARTIFACT_FILES["sa2"], index=False)
    od.to_csv(artifact_dir / RAW_ARTIFACT_FILES["od"], index=False, encoding="utf-8", lineterminator="\n")

    return {"places": places, "roads": roads, "sa2": sa2, "od": od, "source_mode": "refreshed_source_artifacts"}


def raw_artifacts_available(artifact_dir: Path) -> bool:
    return all((artifact_dir / filename).exists() for filename in RAW_ARTIFACT_FILES.values())


def legacy_exports_available(artifact_dir: Path) -> bool:
    return all((artifact_dir / filename).exists() for filename in LEGACY_EXPORT_FILES.values())


def load_public_inputs(artifact_dir: Path) -> dict[str, object]:
    if REFRESH_PUBLIC_DEMO_SOURCES:
        return refresh_public_source_artifacts(artifact_dir, output_crs=OUTPUT_CRS)
    if public_artifacts_available(artifact_dir):
        return load_public_artifacts(artifact_dir)

    expected_public = ", ".join(PUBLIC_ARTIFACT_FILES.values())
    raise FileNotFoundError(
        "No prepared public demo artifacts were found. "
        f"Set PBI_DECKGL_ARTIFACT_DIR to a folder containing public artifacts ({expected_public}), "
        "or set REFRESH_PUBLIC_DEMO_SOURCES=true to download NZTA, Stats NZ boundary, and GTFS sources."
    )


def load_raw_artifacts(artifact_dir: Path) -> dict[str, object]:
    return {
        "places": gpd.read_parquet(artifact_dir / RAW_ARTIFACT_FILES["places"]),
        "roads": gpd.read_parquet(artifact_dir / RAW_ARTIFACT_FILES["roads"]),
        "sa2": gpd.read_parquet(artifact_dir / RAW_ARTIFACT_FILES["sa2"]),
        "od": pd.read_csv(artifact_dir / RAW_ARTIFACT_FILES["od"], low_memory=False),
        "source_mode": "prepared_source_artifacts",
    }


def load_legacy_exports(artifact_dir: Path) -> dict[str, object]:
    return {
        "places": pd.read_csv(artifact_dir / LEGACY_EXPORT_FILES["places"], low_memory=False),
        "roads": pd.read_csv(artifact_dir / LEGACY_EXPORT_FILES["roads"], low_memory=False),
        "place_surface_fact": pd.read_csv(artifact_dir / LEGACY_EXPORT_FILES["place_surface_fact"], low_memory=False),
        "place_road_bridge": pd.read_csv(artifact_dir / LEGACY_EXPORT_FILES["place_road_bridge"], low_memory=False),
        "road_surface_dimension": pd.read_csv(
            artifact_dir / LEGACY_EXPORT_FILES["road_surface_dimension"], low_memory=False
        ),
        "sa2_reference": pd.read_csv(artifact_dir / LEGACY_EXPORT_FILES["sa2_reference"], low_memory=False),
        "od_arc": pd.read_csv(artifact_dir / LEGACY_EXPORT_FILES["od_arc"], low_memory=False),
        "source_mode": "legacy_power_bi_exports",
    }


def load_inputs() -> dict[str, object]:
    if REFRESH_FROM_SOURCE:
        inputs = refresh_source_artifacts(ARTIFACT_DIR)
    elif raw_artifacts_available(ARTIFACT_DIR):
        inputs = load_raw_artifacts(ARTIFACT_DIR)
    elif legacy_exports_available(ARTIFACT_DIR):
        inputs = load_legacy_exports(ARTIFACT_DIR)
    else:
        expected_raw = ", ".join(RAW_ARTIFACT_FILES.values())
        expected_legacy = ", ".join(LEGACY_EXPORT_FILES.values())
        raise FileNotFoundError(
            "No prepared artifacts were found. "
            f"Set PBI_DECKGL_ARTIFACT_DIR to a folder containing either raw artifacts ({expected_raw}) "
            f"or legacy generated exports ({expected_legacy}), or set REFRESH_FROM_SOURCE=true with API keys."
        )

    inputs.update(load_public_inputs(ARTIFACT_DIR))
    return inputs


def require_columns(df: pd.DataFrame, columns: list[str], table_name: str) -> None:
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")


def first_existing_column(df: pd.DataFrame, candidates: tuple[str, ...]) -> str | None:
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    return None


def coalesce_columns(df: pd.DataFrame, candidates: tuple[str, ...], default: object = pd.NA) -> pd.Series:
    existing = [candidate for candidate in candidates if candidate in df.columns]
    if not existing:
        return pd.Series(default, index=df.index)
    result = df[existing[0]]
    for column in existing[1:]:
        result = result.fillna(df[column])
    return result


def clean_string(series: pd.Series, *, default: str | None = None) -> pd.Series:
    result = series.astype("string").str.strip()
    result = result.mask(result.eq(""), pd.NA)
    if default is not None:
        result = result.fillna(default)
    return result


def normalize_bool(series: pd.Series, default: bool = False) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(default).astype(bool)
    lowered = series.astype("string").str.strip().str.lower()
    mapped = lowered.map({"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False})
    return mapped.fillna(default).astype(bool)


def simplify_for_display(gdf: gpd.GeoDataFrame, tolerance_m: float) -> gpd.GeoDataFrame:
    display = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    if display.crs is None:
        display = display.set_crs(ANALYSIS_CRS)
    if tolerance_m <= 0:
        return display.to_crs(OUTPUT_CRS)
    metric = display.to_crs(ANALYSIS_CRS).copy()
    metric.geometry = metric.geometry.simplify(tolerance=tolerance_m, preserve_topology=True)
    metric = metric[metric.geometry.notna() & ~metric.geometry.is_empty].copy()
    return metric.to_crs(OUTPUT_CRS)


def representative_points_lonlat(gdf: gpd.GeoDataFrame) -> tuple[pd.Series, pd.Series]:
    metric = gdf.to_crs(ANALYSIS_CRS)
    points = gpd.GeoSeries(metric.geometry.representative_point(), crs=ANALYSIS_CRS).to_crs(OUTPUT_CRS)
    return points.x, points.y


def encode_wkp_geometry(geom) -> object:
    if geom is None or geom.is_empty:
        return pd.NA
    return wkp.encode(wkp_context, geom, precision=WKP_PRECISION).decode("ascii")


def add_geometry_export_columns(gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    export = pd.DataFrame(gdf.drop(columns="geometry"))
    export["wkp"] = gdf.geometry.apply(encode_wkp_geometry)
    if EXPORT_WKT_DEBUG:
        export["wkt"] = gdf.geometry.to_wkt(rounding_precision=WKP_PRECISION)
    return export


def rank_desc_nullable(series: pd.Series, mask: pd.Series | None = None) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    valid = values.notna()
    if mask is not None:
        valid = valid & mask.fillna(False)
    result = pd.Series(pd.NA, index=series.index, dtype="Int64")
    if valid.any():
        result.loc[valid] = values.loc[valid].rank(method="first", ascending=False).astype("Int64")
    return result


def log_scaled_width(series: pd.Series, *, min_width: int, max_width: int) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    positive = values[values > 0]
    if positive.empty:
        return pd.Series(0, index=series.index, dtype="Int64")
    log_min = math.log1p(float(positive.min()))
    log_max = math.log1p(float(positive.max()))
    log_range = max(log_max - log_min, 1e-9)

    def width(value) -> int:
        if pd.isna(value) or value <= 0:
            return 0
        normalized = (math.log1p(float(value)) - log_min) / log_range
        normalized = max(0.0, min(1.0, normalized))
        return int(round(min_width + (max_width - min_width) * normalized, 0))

    return values.apply(width).astype("Int64")


def power_bi_field_name(value: str) -> str:
    return re.sub(r"[^0-9a-zA-Z]+", "_", str(value)).strip("_").lower()


def hex_to_rgb(value: str) -> tuple[int, int, int]:
    hex_value = value.strip().lstrip("#")[:6]
    if len(hex_value) == 3:
        hex_value = "".join(channel * 2 for channel in hex_value)
    return tuple(int(hex_value[index : index + 2], 16) for index in (0, 2, 4))


def mix_hex_colors(base_color: str, mix_color: str = "#fff4cf", mix_weight: float = 0.35) -> str:
    base_rgb = hex_to_rgb(base_color)
    mix_rgb = hex_to_rgb(mix_color)
    blended = [round(base_rgb[index] * (1 - mix_weight) + mix_rgb[index] * mix_weight) for index in range(3)]
    return "#" + "".join(f"{channel:02x}" for channel in blended)


def rgba_string(hex_color: str, alpha: float) -> str:
    red, green, blue = hex_to_rgb(hex_color)
    return f"rgba({red}, {green}, {blue}, {alpha:.3f})"


def arc_count_band(value) -> str:
    if pd.isna(value) or value <= 0:
        return "No flow"
    if value >= 200:
        return "Very high (200+)"
    if value >= 85:
        return "High (85-199)"
    if value >= 52:
        return "Elevated (52-84)"
    if value >= 25:
        return "Medium (25-51)"
    return "Low (6-24)"


def write_csv(df: pd.DataFrame, filename: str, output_dir: Path = POWER_BI_OUTPUT_DIR) -> dict[str, object]:
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / filename
    df.to_csv(path, index=False, encoding="utf-8", lineterminator="\n")
    return {
        "filename": filename,
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "size_bytes": int(path.stat().st_size),
    }


## Load Prepared Artifacts Or Refresh APIs

The default branch is offline. It first looks for raw GeoParquet/CSV artifacts, then falls back to the previous generated CSV exports. API refresh runs only when `REFRESH_FROM_SOURCE=true`.


In [ ]:
inputs = load_inputs()
SOURCE_MODE = inputs["source_mode"]
PUBLIC_SOURCE_MODE = inputs["public_source_mode"]

print(f"Loaded inputs using source mode: {SOURCE_MODE}")
print(f"Loaded public demo inputs using source mode: {PUBLIC_SOURCE_MODE}")
if SOURCE_MODE == "legacy_power_bi_exports":
    print("Using existing generated CSVs as the offline artifact source.")
else:
    print("Using source-level artifacts with geospatial processing enabled.")


## Build Polygon And Path Tables

When source-level artifacts are available, analytic values are computed in NZTM before display simplification. When only legacy generated CSVs are available, the notebook reuses their existing WKP geometry and records that in the manifest.


In [ ]:
ROAD_SURFACE_COLORS = {
    "sealed": "#1464a5cc",
    "metalled": "#bb7f22cc",
    "unmetalled": "#734a12cc",
    "unsealed": "#bb7f22cc",
    "4wd": "#734a12cc",
    "Unknown": "#8b8f94cc",
}


def build_places_roads_from_source(
    places_raw: gpd.GeoDataFrame,
    roads_raw: gpd.GeoDataFrame,
) -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, object]]:
    places_nztm = places_raw.to_crs(ANALYSIS_CRS)
    roads_raw_nztm = roads_raw.to_crs(ANALYSIS_CRS)

    places_for_model = places_nztm[places_nztm["type"].isin(LINZ_PLACE_TYPES_FOR_MODEL)].copy()
    places_for_model["place_feature_id"] = places_for_model["id"]
    places_for_model["place_name"] = coalesce_columns(places_for_model, ("name_ascii", "name"))
    places_for_model["place_type"] = places_for_model["type"]
    places_for_model["territorial_authority_name"] = coalesce_columns(
        places_for_model,
        ("territorial_authority_ascii", "territorial_authority"),
    )

    place_dimension_nztm = places_for_model[
        ["place_feature_id", "place_name", "place_type", "territorial_authority_name", "geometry"]
    ].copy()
    place_dimension_nztm["place_area_m2"] = place_dimension_nztm.geometry.area
    place_dimension_nztm["place_area_km2"] = place_dimension_nztm["place_area_m2"] / 1_000_000

    road_name = coalesce_columns(roads_raw_nztm, ("name_ascii", "name"))
    road_id_column = first_existing_column(roads_raw_nztm, ("t50_fid", "id", "fid"))
    if road_id_column is None:
        roads_raw_nztm = roads_raw_nztm.reset_index().rename(columns={"index": "_generated_road_id"})
        road_id_column = "_generated_road_id"

    roads_nztm = roads_raw_nztm[[road_id_column, "surface", "geometry"]].copy()
    roads_nztm = roads_nztm.rename(columns={road_id_column: "road_feature_id", "surface": "road_surface"})
    roads_nztm["road_name"] = clean_string(road_name)
    roads_nztm["road_surface"] = clean_string(roads_nztm["road_surface"], default="Unknown")
    roads_nztm["road_length_total_km"] = roads_nztm.geometry.length / 1000

    road_place_pairs = gpd.sjoin(
        roads_nztm,
        place_dimension_nztm[["place_feature_id", "geometry"]],
        how="inner",
        predicate="intersects",
    )
    place_geometries = place_dimension_nztm.geometry.rename("place_geometry")
    road_place_pairs = road_place_pairs.join(place_geometries, on="index_right")
    road_place_pairs["road_length_m"] = road_place_pairs.geometry.intersection(
        gpd.GeoSeries(road_place_pairs["place_geometry"], crs=roads_nztm.crs)
    ).length
    road_place_pairs = road_place_pairs[road_place_pairs["road_length_m"] > 0].copy()
    road_place_pairs["road_length_km"] = road_place_pairs["road_length_m"] / 1000

    place_road_bridge = road_place_pairs.groupby(
        ["place_feature_id", "road_feature_id", "road_surface"],
        as_index=False,
    )["road_length_km"].sum()
    place_road_bridge = place_road_bridge.rename(columns={"road_length_km": "road_length_in_place_km"})
    place_road_bridge = place_road_bridge.merge(
        roads_nztm[["road_feature_id", "road_length_total_km"]],
        on="road_feature_id",
        how="left",
    )
    place_road_bridge["road_length_share_of_road"] = (
        place_road_bridge["road_length_in_place_km"]
        .div(place_road_bridge["road_length_total_km"].where(place_road_bridge["road_length_total_km"] > 0))
        .fillna(0)
    )

    road_stats_by_place_surface = (
        place_road_bridge.groupby(["place_feature_id", "road_surface"], as_index=False)["road_length_in_place_km"]
        .sum()
        .rename(columns={"road_length_in_place_km": "road_length_km"})
    )

    road_surface_dimension = pd.DataFrame({"road_surface": sorted(roads_nztm["road_surface"].unique())})
    road_surface_dimension["road_surface_sort_order"] = range(1, len(road_surface_dimension) + 1)

    place_surface_scaffold = (
        place_dimension_nztm[["place_feature_id"]]
        .assign(_join_key=1)
        .merge(road_surface_dimension[["road_surface"]].assign(_join_key=1), on="_join_key", how="inner")
        .drop(columns="_join_key")
    )
    place_surface_density_fact = place_surface_scaffold.merge(
        road_stats_by_place_surface,
        on=["place_feature_id", "road_surface"],
        how="left",
    ).fillna({"road_length_km": 0})
    place_surface_density_fact = place_surface_density_fact.merge(
        place_dimension_nztm[["place_feature_id", "place_area_km2"]],
        on="place_feature_id",
        how="left",
    )
    place_surface_density_fact["road_density_km_per_km2"] = (
        place_surface_density_fact["road_length_km"]
        .div(place_surface_density_fact["place_area_km2"].where(place_surface_density_fact["place_area_km2"] > 0))
        .fillna(0)
    )

    road_stats_by_place = (
        place_road_bridge.groupby("place_feature_id", as_index=False)["road_length_in_place_km"]
        .sum()
        .rename(columns={"road_length_in_place_km": "road_length_total_km"})
    )
    place_dimension_nztm = place_dimension_nztm.merge(
        road_stats_by_place,
        on="place_feature_id",
        how="left",
    ).fillna({"road_length_total_km": 0})
    place_dimension_nztm["road_density_km_per_km2"] = (
        place_dimension_nztm["road_length_total_km"]
        .div(place_dimension_nztm["place_area_km2"].where(place_dimension_nztm["place_area_km2"] > 0))
        .fillna(0)
    )

    diagnostics = {
        "linz_place_type_counts": places_nztm["type"].value_counts(dropna=False).to_dict(),
        "selected_place_type_counts": places_for_model["type"].value_counts(dropna=False).to_dict(),
    }
    return (
        place_dimension_nztm,
        roads_nztm,
        place_surface_density_fact,
        place_road_bridge,
        road_surface_dimension,
        diagnostics,
    )


def prepare_place_polygon_from_source(place_source: gpd.GeoDataFrame) -> tuple[pd.DataFrame, gpd.GeoDataFrame]:
    require_columns(
        place_source,
        [
            "place_feature_id",
            "place_name",
            "place_type",
            "territorial_authority_name",
            "place_area_km2",
            "road_length_total_km",
            "road_density_km_per_km2",
        ],
        "place source",
    )
    display_gdf = simplify_for_display(place_source, PLACE_SIMPLIFY_TOLERANCE_M)
    label_lon, label_lat = representative_points_lonlat(display_gdf)
    display_gdf["geometry_id"] = "place_" + display_gdf["place_feature_id"].astype("string").str.strip()
    display_gdf["layer_type"] = "polygon"
    display_gdf["label_lon"] = label_lon.values
    display_gdf["label_lat"] = label_lat.values
    display_gdf["place_road_density_rank"] = rank_desc_nullable(display_gdf["road_density_km_per_km2"])
    display_gdf["polygon_line_width_m"] = 50
    display_gdf["polygon_line_color_hex"] = "#2f3437cc"
    display_gdf["polygon_fill_color_value"] = display_gdf["road_density_km_per_km2"]
    display_gdf["polygon_extrude_elevation_m"] = (
        (pd.to_numeric(display_gdf["road_density_km_per_km2"], errors="coerce").fillna(0).clip(upper=20) * 75)
        .round(0)
        .astype("Int64")
    )
    export = add_geometry_export_columns(display_gdf)
    return order_place_polygon(export), display_gdf


def prepare_road_path_from_source(road_source: gpd.GeoDataFrame) -> tuple[pd.DataFrame, gpd.GeoDataFrame]:
    require_columns(road_source, ["road_feature_id", "road_surface", "road_length_total_km"], "road source")
    display_gdf = simplify_for_display(road_source, ROAD_SIMPLIFY_TOLERANCE_M)
    display_gdf["road_surface"] = clean_string(display_gdf["road_surface"], default="Unknown")
    display_gdf["road_name"] = clean_string(display_gdf.get("road_name", pd.Series(pd.NA, index=display_gdf.index)))
    display_gdf["geometry_id"] = "road_" + display_gdf["road_feature_id"].astype("string").str.strip()
    display_gdf["layer_type"] = "path"
    display_gdf["road_length_rank"] = rank_desc_nullable(display_gdf["road_length_total_km"])
    display_gdf["path_width_m"] = log_scaled_width(display_gdf["road_length_total_km"], min_width=20, max_width=180)
    surface_color = display_gdf["road_surface"].map(ROAD_SURFACE_COLORS).fillna(ROAD_SURFACE_COLORS["Unknown"])
    display_gdf["path_color_hex"] = surface_color
    export = add_geometry_export_columns(display_gdf)
    return order_road_path(export), display_gdf


def order_place_polygon(df: pd.DataFrame) -> pd.DataFrame:
    ordered_columns = [
        "geometry_id",
        "layer_type",
        "place_feature_id",
        "place_name",
        "place_type",
        "territorial_authority_name",
        "place_area_m2",
        "place_area_km2",
        "road_length_total_km",
        "road_density_km_per_km2",
        "place_road_density_rank",
        "label_lon",
        "label_lat",
        "polygon_line_width_m",
        "polygon_line_color_hex",
        "polygon_fill_color_value",
        "polygon_extrude_elevation_m",
        "wkp",
    ]
    if EXPORT_WKT_DEBUG and "wkt" in df.columns:
        ordered_columns.append("wkt")
    return (
        df[[column for column in ordered_columns if column in df.columns]]
        .sort_values(["place_road_density_rank", "place_name"], na_position="last")
        .reset_index(drop=True)
    )


def order_road_path(df: pd.DataFrame) -> pd.DataFrame:
    ordered_columns = [
        "geometry_id",
        "layer_type",
        "road_feature_id",
        "road_surface",
        "road_name",
        "road_length_total_km",
        "road_length_rank",
        "path_width_m",
        "path_color_hex",
        "wkp",
    ]
    if EXPORT_WKT_DEBUG and "wkt" in df.columns:
        ordered_columns.append("wkt")
    return (
        df[[column for column in ordered_columns if column in df.columns]]
        .sort_values(["road_length_rank", "road_feature_id"], na_position="last")
        .reset_index(drop=True)
    )


def prepare_place_polygon_from_legacy(df: pd.DataFrame) -> tuple[pd.DataFrame, None]:
    require_columns(
        df,
        [
            "place_feature_id",
            "place_name",
            "place_type",
            "territorial_authority_name",
            "place_area_km2",
            "road_length_total_km",
            "road_density_km_per_km2",
            "geometry_wkp",
        ],
        "legacy place export",
    )
    output = df.copy()
    output["geometry_id"] = "place_" + output["place_feature_id"].astype("string").str.strip()
    output["layer_type"] = "polygon"
    output["wkp"] = output["geometry_wkp"]
    if "label_lon" not in output.columns:
        output["label_lon"] = pd.NA
    if "label_lat" not in output.columns:
        output["label_lat"] = pd.NA
    output["place_road_density_rank"] = rank_desc_nullable(output["road_density_km_per_km2"])
    output["polygon_line_width_m"] = 50
    output["polygon_line_color_hex"] = "#2f3437cc"
    output["polygon_fill_color_value"] = output["road_density_km_per_km2"]
    output["polygon_extrude_elevation_m"] = (
        (pd.to_numeric(output["road_density_km_per_km2"], errors="coerce").fillna(0).clip(upper=20) * 75)
        .round(0)
        .astype("Int64")
    )
    return order_place_polygon(output), None


def prepare_road_path_from_legacy(df: pd.DataFrame) -> tuple[pd.DataFrame, None]:
    require_columns(
        df, ["road_feature_id", "road_surface", "road_length_total_km", "geometry_wkp"], "legacy road export"
    )
    output = df.copy()
    output["road_surface"] = clean_string(output["road_surface"], default="Unknown")
    output["road_name"] = clean_string(output.get("road_name", pd.Series(pd.NA, index=output.index)))
    output["geometry_id"] = "road_" + output["road_feature_id"].astype("string").str.strip()
    output["layer_type"] = "path"
    output["road_length_rank"] = rank_desc_nullable(output["road_length_total_km"])
    output["path_width_m"] = log_scaled_width(output["road_length_total_km"], min_width=20, max_width=180)
    surface_color = output["road_surface"].map(ROAD_SURFACE_COLORS).fillna(ROAD_SURFACE_COLORS["Unknown"])
    output["path_color_hex"] = surface_color
    output["wkp"] = output["geometry_wkp"]
    return order_road_path(output), None


if SOURCE_MODE in {"refreshed_source_artifacts", "prepared_source_artifacts"}:
    (
        place_model_nztm,
        road_model_nztm,
        place_surface_density_fact,
        place_road_bridge,
        road_surface_dimension,
        place_road_diagnostics,
    ) = build_places_roads_from_source(inputs["places"], inputs["roads"])
    place_polygon, place_polygon_gdf = prepare_place_polygon_from_source(place_model_nztm)
    road_path, road_path_gdf = prepare_road_path_from_source(road_model_nztm)
else:
    place_surface_density_fact = inputs["place_surface_fact"].copy()
    place_road_bridge = inputs["place_road_bridge"].copy()
    road_surface_dimension = inputs["road_surface_dimension"].copy()
    place_road_diagnostics = {
        "legacy_exports_used": True,
        "display_simplification_note": "Existing WKP geometry was reused from legacy CSV exports.",
    }
    place_polygon, place_polygon_gdf = prepare_place_polygon_from_legacy(inputs["places"])
    road_path, road_path_gdf = prepare_road_path_from_legacy(inputs["roads"])

print(f"Place polygon rows: {len(place_polygon):,}")
print(f"Road path rows: {len(road_path):,}")
print(f"Place-surface fact rows: {len(place_surface_density_fact):,}")
print(f"Place-road bridge rows: {len(place_road_bridge):,}")
print(f"Road surface rows: {len(road_surface_dimension):,}")
if len(road_path) > POWER_BI_ROW_WINDOW:
    print(
        f"Power BI row window warning: filter nz_road_path to road_length_rank <= {POWER_BI_ROW_WINDOW:,} or use slicers."
    )


## Build SA2 Reference And Arc Table

Arc geometry is stored as four coordinate fields because the visual's arc parser ignores WKT/WKP and requires point 1 and point 2 latitude/longitude fields.


In [ ]:
def build_sa2_reference_from_source(sa2_source: gpd.GeoDataFrame) -> pd.DataFrame:
    sa2_nztm = sa2_source.to_crs(ANALYSIS_CRS)
    code_column = "SA22023_V1_00"
    require_columns(sa2_nztm, [code_column, "LAND_AREA_SQ_KM", "AREA_SQ_KM", "geometry"], "SA2 source")
    sa2_nztm = sa2_nztm.copy()
    sa2_nztm["sa2_code"] = clean_string(sa2_nztm[code_column])
    sa2_nztm["sa2_name"] = coalesce_columns(sa2_nztm, ("SA22023_V1_00_NAME_ASCII", "SA22023_V1_00_NAME"))
    sa2_nztm = sa2_nztm.drop_duplicates(subset="sa2_code")
    centroids = gpd.GeoSeries(sa2_nztm.geometry.centroid, crs=ANALYSIS_CRS).to_crs(OUTPUT_CRS)
    return (
        pd.DataFrame(
            {
                "geometry_id": "sa2_" + sa2_nztm["sa2_code"].astype("string"),
                "sa2_code": sa2_nztm["sa2_code"].astype("string"),
                "sa2_name": sa2_nztm["sa2_name"].astype("string"),
                "center_lat": centroids.y,
                "center_lon": centroids.x,
                "land_area_sq_km": pd.to_numeric(sa2_nztm["LAND_AREA_SQ_KM"], errors="coerce"),
                "area_sq_km": pd.to_numeric(sa2_nztm["AREA_SQ_KM"], errors="coerce"),
                "has_geometry": sa2_nztm.geometry.notna() & ~sa2_nztm.geometry.is_empty,
            }
        )
        .sort_values("sa2_code")
        .reset_index(drop=True)
    )


def normalize_sa2_reference(reference: pd.DataFrame) -> pd.DataFrame:
    require_columns(
        reference, ["geometry_id", "sa2_code", "sa2_name", "center_lat", "center_lon", "has_geometry"], "SA2 reference"
    )
    output = reference.copy()
    output["sa2_code"] = clean_string(output["sa2_code"])
    output["geometry_id"] = "sa2_" + output["sa2_code"].astype("string")
    output["center_lat"] = pd.to_numeric(output["center_lat"], errors="coerce")
    output["center_lon"] = pd.to_numeric(output["center_lon"], errors="coerce")
    output["has_geometry"] = normalize_bool(output["has_geometry"])
    for column in ["land_area_sq_km", "area_sq_km"]:
        if column in output.columns:
            output[column] = pd.to_numeric(output[column], errors="coerce")
    ordered = [
        "geometry_id",
        "sa2_code",
        "sa2_name",
        "center_lat",
        "center_lon",
        "land_area_sq_km",
        "area_sq_km",
        "has_geometry",
    ]
    return (
        output[[column for column in ordered if column in output.columns]]
        .sort_values("sa2_code")
        .reset_index(drop=True)
    )


def build_arc_table_from_raw_od(od_source: pd.DataFrame, sa2_reference: pd.DataFrame) -> pd.DataFrame:
    residence_code_column = "SA22023_V1_00_usual_residence_address"
    workplace_code_column = "SA22023_V1_00_workplace_address"
    require_columns(od_source, [residence_code_column, workplace_code_column], "travel-to-work OD source")
    residence_name_columns = [
        column
        for column in ("SA22023_V1_00_NAME_ASCII_usual_residence_address", "SA22023_V1_00_NAME_usual_residence_address")
        if column in od_source.columns
    ]
    workplace_name_columns = [
        column
        for column in ("SA22023_V1_00_NAME_ASCII_workplace_address", "SA22023_V1_00_NAME_workplace_address")
        if column in od_source.columns
    ]
    od_2023_count_columns = [column for column in od_source.columns if str(column).startswith("2023_")]
    od = od_source[
        [
            residence_code_column,
            workplace_code_column,
            *residence_name_columns,
            *workplace_name_columns,
            *od_2023_count_columns,
        ]
    ].copy()
    od = od.rename(
        columns={
            residence_code_column: "origin_sa2_code",
            workplace_code_column: "destination_sa2_code",
            **{column: f"count_{power_bi_field_name(column)}" for column in od_2023_count_columns},
        }
    )
    od["origin_sa2_name"] = coalesce_columns(od, tuple(residence_name_columns))
    od["destination_sa2_name"] = coalesce_columns(od, tuple(workplace_name_columns))
    od = od.drop(columns=[*residence_name_columns, *workplace_name_columns], errors="ignore")
    return prepare_arc_table(od, sa2_reference)


def prepare_arc_table(od_source: pd.DataFrame, sa2_reference: pd.DataFrame | None) -> pd.DataFrame:
    od = od_source.copy()
    require_columns(od, ["origin_sa2_code", "destination_sa2_code"], "arc source")
    od["origin_sa2_code"] = clean_string(od["origin_sa2_code"])
    od["destination_sa2_code"] = clean_string(od["destination_sa2_code"])

    count_columns = [column for column in od.columns if str(column).startswith("count_2023_")]
    for column in count_columns:
        od[column] = pd.to_numeric(od[column], errors="coerce").round().astype("Int64")
        od.loc[od[column] < 0, column] = pd.NA

    if "people_count" not in od.columns:
        if "count_2023_total_stated" not in od.columns:
            raise ValueError("Arc source requires people_count or count_2023_total_stated.")
        od["people_count"] = od["count_2023_total_stated"]
    od["people_count"] = pd.to_numeric(od["people_count"], errors="coerce").round().astype("Int64")

    if sa2_reference is not None:
        sa2_reference = normalize_sa2_reference(sa2_reference)
        origin_lookup = sa2_reference.rename(
            columns={
                "geometry_id": "origin_geometry_id",
                "sa2_code": "origin_sa2_code",
                "sa2_name": "origin_sa2_reference_name",
                "center_lat": "point1_latitude",
                "center_lon": "point1_longitude",
                "has_geometry": "origin_has_geometry",
            }
        )[
            [
                "origin_geometry_id",
                "origin_sa2_code",
                "origin_sa2_reference_name",
                "point1_latitude",
                "point1_longitude",
                "origin_has_geometry",
            ]
        ]
        destination_lookup = sa2_reference.rename(
            columns={
                "geometry_id": "destination_geometry_id",
                "sa2_code": "destination_sa2_code",
                "sa2_name": "destination_sa2_reference_name",
                "center_lat": "point2_latitude",
                "center_lon": "point2_longitude",
                "has_geometry": "destination_has_geometry",
            }
        )[
            [
                "destination_geometry_id",
                "destination_sa2_code",
                "destination_sa2_reference_name",
                "point2_latitude",
                "point2_longitude",
                "destination_has_geometry",
            ]
        ]

        drop_origin = [
            column for column in origin_lookup.columns if column in od.columns and column != "origin_sa2_code"
        ]
        drop_destination = [
            column for column in destination_lookup.columns if column in od.columns and column != "destination_sa2_code"
        ]
        od = od.drop(columns=[*drop_origin, *drop_destination], errors="ignore")
        od = od.merge(origin_lookup, on="origin_sa2_code", how="left")
        od = od.merge(destination_lookup, on="destination_sa2_code", how="left")

    for column in [
        "origin_sa2_name",
        "origin_sa2_reference_name",
        "destination_sa2_name",
        "destination_sa2_reference_name",
    ]:
        if column not in od.columns:
            od[column] = pd.NA
    for column in ["point1_latitude", "point1_longitude", "point2_latitude", "point2_longitude"]:
        od[column] = pd.to_numeric(od.get(column, pd.Series(pd.NA, index=od.index)), errors="coerce")
    for column in ["origin_has_geometry", "destination_has_geometry"]:
        od[column] = normalize_bool(od.get(column, pd.Series(False, index=od.index)))

    od["has_origin_point"] = od["point1_latitude"].notna() & od["point1_longitude"].notna()
    od["has_destination_point"] = od["point2_latitude"].notna() & od["point2_longitude"].notna()
    od["has_both_points"] = od["has_origin_point"] & od["has_destination_point"]
    od["is_same_sa2"] = od["origin_sa2_code"].eq(od["destination_sa2_code"])
    od["arc_is_valid"] = od["has_both_points"] & ~od["is_same_sa2"] & od["people_count"].gt(0).fillna(False)
    od["people_count_rank"] = rank_desc_nullable(od["people_count"], od["arc_is_valid"])

    valid_people_count = od.loc[od["arc_is_valid"], "people_count"].dropna().astype(float)
    if valid_people_count.empty:
        color_scale = None
        log_people_min = 0.0
        log_people_range = 1.0
    else:
        color_min = float(valid_people_count.min())
        color_max = float(valid_people_count.max())
        color_scale = linear.YlOrRd_09.scale(color_min, color_max if color_max > color_min else color_min + 1)
        log_people_min = math.log1p(color_min)
        log_people_max = math.log1p(color_max)
        log_people_range = max(log_people_max - log_people_min, 1e-9)

    def normalized_people_count(value) -> float:
        if pd.isna(value) or value <= 0:
            return 0.0
        normalized = (math.log1p(float(value)) - log_people_min) / log_people_range
        return max(0.0, min(1.0, normalized))

    od["arc_count_band"] = od["people_count"].apply(arc_count_band)
    band_sort_order = {
        "No flow": 0,
        "Low (6-24)": 1,
        "Medium (25-51)": 2,
        "Elevated (52-84)": 3,
        "High (85-199)": 4,
        "Very high (200+)": 5,
    }
    od["arc_count_band_sort_order"] = od["arc_count_band"].map(band_sort_order).astype("Int64")
    od["arc_line_width_m"] = (
        od["people_count"]
        .apply(
            lambda value: (
                0 if normalized_people_count(value) <= 0 else int(round(50 + 450 * normalized_people_count(value), 0))
            )
        )
        .astype("Int64")
    )
    od["arc_opacity"] = od["people_count"].apply(
        lambda value: (
            0.0 if normalized_people_count(value) <= 0 else round(0.18 + 0.77 * normalized_people_count(value), 3)
        )
    )
    od.loc[~od["arc_is_valid"], "arc_line_width_m"] = 0
    od.loc[~od["arc_is_valid"], "arc_opacity"] = 0.0
    od["arc_source_color_value"] = od["people_count"]
    od["arc_target_color_value"] = od["people_count"]
    od["arc_base_color_hex"] = od["people_count"].apply(
        lambda value: (
            color_scale(float(value)) if color_scale is not None and pd.notna(value) and value > 0 else "#d9d9d9"
        )
    )
    od["arc_source_color_rgba"] = od.apply(
        lambda row: rgba_string(
            mix_hex_colors(row["arc_base_color_hex"], mix_weight=0.45),
            max(0.0, round(row["arc_opacity"] * 0.72, 3)),
        ),
        axis=1,
    )
    od["arc_target_color_rgba"] = od.apply(
        lambda row: rgba_string(row["arc_base_color_hex"], row["arc_opacity"]),
        axis=1,
    )

    base_id = (
        "sa2_od_"
        + od["origin_sa2_code"].fillna("unknown_origin")
        + "__"
        + od["destination_sa2_code"].fillna("unknown_destination")
    )
    sequence = base_id.groupby(base_id).cumcount()
    od["geometry_id"] = base_id.where(sequence.eq(0), base_id + "__" + (sequence + 1).astype("string"))
    od["layer_type"] = "arc"

    ordered_columns = [
        "geometry_id",
        "layer_type",
        "origin_geometry_id",
        "origin_sa2_code",
        "origin_sa2_name",
        "origin_sa2_reference_name",
        "destination_geometry_id",
        "destination_sa2_code",
        "destination_sa2_name",
        "destination_sa2_reference_name",
        "point1_latitude",
        "point1_longitude",
        "point2_latitude",
        "point2_longitude",
        "people_count",
        "people_count_rank",
        "arc_line_width_m",
        "arc_opacity",
        "arc_count_band",
        "arc_count_band_sort_order",
        "arc_source_color_value",
        "arc_target_color_value",
        "arc_base_color_hex",
        "arc_source_color_rgba",
        "arc_target_color_rgba",
        *count_columns,
        "origin_has_geometry",
        "destination_has_geometry",
        "has_origin_point",
        "has_destination_point",
        "has_both_points",
        "is_same_sa2",
        "arc_is_valid",
    ]
    for column in ordered_columns:
        if column not in od.columns:
            od[column] = pd.NA
    return (
        od[ordered_columns]
        .sort_values(
            ["arc_is_valid", "people_count_rank", "origin_sa2_code", "destination_sa2_code"],
            ascending=[False, True, True, True],
            na_position="last",
        )
        .reset_index(drop=True)
    )


if SOURCE_MODE in {"refreshed_source_artifacts", "prepared_source_artifacts"}:
    sa2_reference = build_sa2_reference_from_source(inputs["sa2"])
    od_arc = build_arc_table_from_raw_od(inputs["od"], sa2_reference)
else:
    sa2_reference = normalize_sa2_reference(inputs["sa2_reference"])
    od_arc = prepare_arc_table(inputs["od_arc"], sa2_reference)

print(f"SA2 reference rows: {len(sa2_reference):,}")
print(f"OD arc rows: {len(od_arc):,}")
print(f"Valid OD arc rows: {int(od_arc['arc_is_valid'].sum()):,}")
if len(od_arc) > POWER_BI_ROW_WINDOW:
    print(
        f"Power BI row window warning: filter arcs to arc_is_valid = TRUE and people_count_rank <= {POWER_BI_ROW_WINDOW:,}."
    )


## Build Point And A-To-B Line Tables

NZTA traffic monitoring sites become point rows. Official AT and Metlink GTFS ferry shapes become straight endpoint-to-endpoint line rows for visual demo coverage.


In [ ]:
traffic_count_site_point = prepare_traffic_count_site_point(inputs["traffic_count_sites"], output_crs=OUTPUT_CRS)
ferry_route_line = prepare_ferry_route_line(inputs["gtfs_feeds"], output_crs=OUTPUT_CRS, analysis_crs=ANALYSIS_CRS)
(
    place_polygon,
    road_path,
    od_arc,
    traffic_count_site_point,
    ferry_route_line,
) = add_visual_tooltips(
    place_polygon=place_polygon,
    road_path=road_path,
    od_arc=od_arc,
    traffic_count_site_point=traffic_count_site_point,
    ferry_route_line=ferry_route_line,
)

print(f"NZTA traffic count site point rows: {len(traffic_count_site_point):,}")
print(f"Ferry A-to-B line rows: {len(ferry_route_line):,}")
if len(ferry_route_line) > POWER_BI_ROW_WINDOW:
    print(f"Power BI row window warning: ferry lines exceed {POWER_BI_ROW_WINDOW:,} rows.")


## Export CSVs, Manifest, And Validation

The visual-ready files use field names that map cleanly to the custom visual roles. The model tables support relationships and slicers in Power BI.


In [ ]:
PRIVATE_PATH_PATTERNS = ["HCCDATA", "fileserver.hcc.govt.nz", "C:\\Users\\TownseD", "\\\\fileserver"]


def validate_unique_id(df: pd.DataFrame, table_name: str) -> list[str]:
    issues = []
    if "geometry_id" not in df.columns:
        issues.append(f"{table_name}: missing geometry_id")
        return issues
    if df["geometry_id"].isna().any():
        issues.append(f"{table_name}: geometry_id contains nulls")
    duplicate_count = int(df["geometry_id"].duplicated().sum())
    if duplicate_count:
        issues.append(f"{table_name}: geometry_id has {duplicate_count:,} duplicates")
    return issues


def validate_layer_type(df: pd.DataFrame, table_name: str, expected: str) -> list[str]:
    if "layer_type" not in df.columns:
        return [f"{table_name}: missing layer_type"]
    actual = set(df["layer_type"].dropna().astype(str).str.lower().unique())
    return (
        [] if actual == {expected} else [f"{table_name}: layer_type values are {sorted(actual)}, expected {expected}"]
    )


def validate_no_private_paths(df: pd.DataFrame, table_name: str) -> list[str]:
    issues = []
    text_columns = [
        column
        for column in df.select_dtypes(include=["object", "string"]).columns
        if column.lower() not in {"wkp", "wkt"}
    ]
    for column in text_columns:
        values = df[column].dropna().astype(str)
        for pattern in PRIVATE_PATH_PATTERNS:
            if values.str.contains(pattern, regex=False).any():
                issues.append(f"{table_name}: column {column} contains private path pattern {pattern!r}")
    return issues


def validate_visual_gdf(gdf: gpd.GeoDataFrame | None, table_name: str, expected_geom_types: set[str]) -> list[str]:
    if gdf is None:
        return []
    issues = []
    epsg = gdf.crs.to_epsg() if gdf.crs is not None else None
    if epsg != 4326:
        issues.append(f"{table_name}: expected EPSG:4326 geometry, found {gdf.crs}")
    actual_types = set(gdf.geometry.geom_type.dropna().unique())
    unexpected = actual_types - expected_geom_types
    if unexpected:
        issues.append(f"{table_name}: unexpected geometry types {sorted(unexpected)}")
    if gdf.geometry.isna().any() or gdf.geometry.is_empty.any():
        issues.append(f"{table_name}: contains null or empty geometry")
    return issues


def validate_wkp_column(df: pd.DataFrame, table_name: str) -> list[str]:
    if "wkp" not in df.columns:
        return [f"{table_name}: missing wkp"]
    if df["wkp"].isna().any() or df["wkp"].astype("string").str.len().fillna(0).eq(0).any():
        return [f"{table_name}: wkp contains blank values"]
    return []


def validate_arc_coordinates(df: pd.DataFrame, table_name: str = "nz_sa2_travel_to_work_od_2023_arc") -> list[str]:
    issues = []
    coordinate_bounds = {
        "point1_latitude": (-90, 90),
        "point2_latitude": (-90, 90),
        "point1_longitude": (-180, 180),
        "point2_longitude": (-180, 180),
    }
    valid_rows = df["arc_is_valid"].fillna(False)
    for column, (lower, upper) in coordinate_bounds.items():
        values = pd.to_numeric(df.loc[valid_rows, column], errors="coerce")
        if values.isna().any() or ~values.between(lower, upper).all():
            issues.append(f"{table_name}: {column} has values outside [{lower}, {upper}] for valid arcs")
    return issues


nz_multigeometry_road_limit = max(0, POWER_BI_ROW_WINDOW - len(place_polygon) - len(traffic_count_site_point))
nz_multigeometry_road_density_map = build_multigeometry_road_density_map(
    place_polygon=place_polygon,
    road_path=road_path,
    traffic_count_site_point=traffic_count_site_point,
    scope_label="nz",
    road_limit=nz_multigeometry_road_limit,
)

visual_tables = {
    "nz_place_polygon.csv": place_polygon,
    "nz_road_path.csv": road_path,
    "nz_sa2_travel_to_work_od_2023_arc.csv": od_arc,
    "nzta_traffic_count_site_point.csv": traffic_count_site_point,
    "nz_ferry_route_line.csv": ferry_route_line,
    "nz_multigeometry_road_density_map.csv": nz_multigeometry_road_density_map,
}
model_tables = {
    "nz_place_surface_density_fact.csv": place_surface_density_fact,
    "nz_place_road_bridge.csv": place_road_bridge,
    "nz_road_surface_dimension.csv": road_surface_dimension,
    "nz_sa2_reference.csv": sa2_reference,
}

hamilton_demo_tables, hamilton_demo_diagnostics = build_hamilton_tla_demo_tables(
    place_polygon=place_polygon,
    road_path=road_path,
    od_arc=od_arc,
    traffic_count_site_point=traffic_count_site_point,
    ferry_route_line=ferry_route_line,
    place_surface_density_fact=place_surface_density_fact,
    place_road_bridge=place_road_bridge,
    road_surface_dimension=road_surface_dimension,
    sa2_reference=sa2_reference,
    territorial_authority_boundary=inputs.get("territorial_authority_boundary"),
    sa2_boundary=inputs.get("sa2_boundary"),
)
hamilton_multigeometry_road_density_map = build_multigeometry_road_density_map(
    place_polygon=hamilton_demo_tables["place_polygon"],
    road_path=hamilton_demo_tables["road_path"],
    traffic_count_site_point=hamilton_demo_tables["traffic_count_site_point"],
    scope_label="hamilton",
)
hamilton_visual_tables = {
    "hamilton_place_polygon.csv": hamilton_demo_tables["place_polygon"],
    "hamilton_road_path.csv": hamilton_demo_tables["road_path"],
    "hamilton_sa2_travel_to_work_od_2023_arc.csv": hamilton_demo_tables["od_arc"],
    "hamilton_nzta_traffic_count_site_point.csv": hamilton_demo_tables["traffic_count_site_point"],
    "hamilton_ferry_route_line.csv": hamilton_demo_tables["ferry_route_line"],
    "hamilton_multigeometry_road_density_map.csv": hamilton_multigeometry_road_density_map,
}
hamilton_model_tables = {
    "hamilton_place_surface_density_fact.csv": hamilton_demo_tables["place_surface_density_fact"],
    "hamilton_place_road_bridge.csv": hamilton_demo_tables["place_road_bridge"],
    "hamilton_road_surface_dimension.csv": hamilton_demo_tables["road_surface_dimension"],
    "hamilton_sa2_reference.csv": hamilton_demo_tables["sa2_reference"],
}

validation_issues = []
hamilton_validation_issues = []
validation_issues.extend(validate_unique_id(place_polygon, "nz_place_polygon"))
validation_issues.extend(validate_unique_id(road_path, "nz_road_path"))
validation_issues.extend(validate_unique_id(od_arc, "nz_sa2_travel_to_work_od_2023_arc"))
validation_issues.extend(validate_unique_id(traffic_count_site_point, "nzta_traffic_count_site_point"))
validation_issues.extend(validate_unique_id(ferry_route_line, "nz_ferry_route_line"))
validation_issues.extend(validate_unique_id(nz_multigeometry_road_density_map, "nz_multigeometry_road_density_map"))
validation_issues.extend(validate_layer_type(place_polygon, "nz_place_polygon", "polygon"))
validation_issues.extend(validate_layer_type(road_path, "nz_road_path", "path"))
validation_issues.extend(validate_layer_type(od_arc, "nz_sa2_travel_to_work_od_2023_arc", "arc"))
validation_issues.extend(validate_layer_type(traffic_count_site_point, "nzta_traffic_count_site_point", "point"))
validation_issues.extend(validate_layer_type(ferry_route_line, "nz_ferry_route_line", "line"))
validation_issues.extend(validate_wkp_column(place_polygon, "nz_place_polygon"))
validation_issues.extend(validate_wkp_column(road_path, "nz_road_path"))
validation_issues.extend(validate_visual_gdf(place_polygon_gdf, "nz_place_polygon", {"Polygon", "MultiPolygon"}))
validation_issues.extend(validate_visual_gdf(road_path_gdf, "nz_road_path", {"LineString", "MultiLineString"}))
validation_issues.extend(validate_arc_coordinates(od_arc))
validation_issues.extend(validate_traffic_point_coordinates(traffic_count_site_point))
validation_issues.extend(validate_ferry_line_coordinates(ferry_route_line))
hamilton_validation_issues.extend(validate_unique_id(hamilton_demo_tables["place_polygon"], "hamilton_place_polygon"))
hamilton_validation_issues.extend(validate_unique_id(hamilton_demo_tables["road_path"], "hamilton_road_path"))
hamilton_validation_issues.extend(
    validate_unique_id(hamilton_demo_tables["od_arc"], "hamilton_sa2_travel_to_work_od_2023_arc")
)
hamilton_validation_issues.extend(
    validate_unique_id(hamilton_demo_tables["traffic_count_site_point"], "hamilton_nzta_traffic_count_site_point")
)
hamilton_validation_issues.extend(
    validate_unique_id(hamilton_demo_tables["ferry_route_line"], "hamilton_ferry_route_line")
)
hamilton_validation_issues.extend(
    validate_unique_id(hamilton_multigeometry_road_density_map, "hamilton_multigeometry_road_density_map")
)
hamilton_validation_issues.extend(
    validate_layer_type(hamilton_demo_tables["place_polygon"], "hamilton_place_polygon", "polygon")
)
hamilton_validation_issues.extend(validate_layer_type(hamilton_demo_tables["road_path"], "hamilton_road_path", "path"))
hamilton_validation_issues.extend(
    validate_layer_type(hamilton_demo_tables["od_arc"], "hamilton_sa2_travel_to_work_od_2023_arc", "arc")
)
hamilton_validation_issues.extend(
    validate_layer_type(
        hamilton_demo_tables["traffic_count_site_point"], "hamilton_nzta_traffic_count_site_point", "point"
    )
)
hamilton_validation_issues.extend(
    validate_layer_type(hamilton_demo_tables["ferry_route_line"], "hamilton_ferry_route_line", "line")
)
hamilton_validation_issues.extend(validate_wkp_column(hamilton_demo_tables["place_polygon"], "hamilton_place_polygon"))
hamilton_validation_issues.extend(validate_wkp_column(hamilton_demo_tables["road_path"], "hamilton_road_path"))
hamilton_validation_issues.extend(
    validate_arc_coordinates(hamilton_demo_tables["od_arc"], "hamilton_sa2_travel_to_work_od_2023_arc")
)
hamilton_validation_issues.extend(validate_traffic_point_coordinates(hamilton_demo_tables["traffic_count_site_point"]))
hamilton_validation_issues.extend(validate_ferry_line_coordinates(hamilton_demo_tables["ferry_route_line"]))
validation_issues.extend(hamilton_validation_issues)
for table_name, df in {**visual_tables, **model_tables}.items():
    validation_issues.extend(validate_no_private_paths(df, table_name))
for table_name, df in {**hamilton_visual_tables, **hamilton_model_tables}.items():
    hamilton_validation_issues.extend(validate_no_private_paths(df, table_name))
validation_issues.extend([issue for issue in hamilton_validation_issues if issue not in validation_issues])

output_file_stats = []
for filename, df in {**visual_tables, **model_tables}.items():
    output_file_stats.append(write_csv(df, filename))

hamilton_output_file_stats = []
for filename, df in {**hamilton_visual_tables, **hamilton_model_tables}.items():
    hamilton_output_file_stats.append(write_csv(df, filename, HAMILTON_DEMO_OUTPUT_DIR))

row_window_warnings = []
if len(place_polygon) > POWER_BI_ROW_WINDOW:
    row_window_warnings.append(
        f"Filter nz_place_polygon to <= {POWER_BI_ROW_WINDOW:,} rows before binding to one visual."
    )
if len(road_path) > POWER_BI_ROW_WINDOW:
    row_window_warnings.append(f"Filter nz_road_path to road_length_rank <= {POWER_BI_ROW_WINDOW:,} or use slicers.")
if len(nz_multigeometry_road_density_map) > POWER_BI_ROW_WINDOW:
    row_window_warnings.append(
        f"Filter nz_multigeometry_road_density_map to <= {POWER_BI_ROW_WINDOW:,} rows before binding to one visual."
    )
valid_arc_rows = int(od_arc["arc_is_valid"].sum())
if valid_arc_rows > POWER_BI_ROW_WINDOW:
    row_window_warnings.append(
        f"Filter nz_sa2_travel_to_work_od_2023_arc to arc_is_valid = TRUE and people_count_rank <= {POWER_BI_ROW_WINDOW:,}."
    )
if len(traffic_count_site_point) > POWER_BI_ROW_WINDOW:
    row_window_warnings.append(
        f"Filter nzta_traffic_count_site_point to <= {POWER_BI_ROW_WINDOW:,} rows before binding to one visual."
    )
valid_ferry_line_rows = int(ferry_route_line["line_is_valid"].sum())
if valid_ferry_line_rows > POWER_BI_ROW_WINDOW:
    row_window_warnings.append(
        f"Filter nz_ferry_route_line to line_is_valid = TRUE and <= {POWER_BI_ROW_WINDOW:,} rows."
    )

diagnostics = {
    **place_road_diagnostics,
    "public_source_mode": PUBLIC_SOURCE_MODE,
    "nzta_traffic_count_site_rows": int(len(traffic_count_site_point)),
    "official_ferry_gtfs_note": "Ferry lines use official Auckland Transport and Metlink GTFS route_type 4 shapes only; private and other NZ ferry routes are not included.",
}

hamilton_row_counts = {
    "hamilton_place_polygon": int(len(hamilton_demo_tables["place_polygon"])),
    "hamilton_road_path": int(len(hamilton_demo_tables["road_path"])),
    "hamilton_sa2_travel_to_work_od_2023_arc": int(len(hamilton_demo_tables["od_arc"])),
    "hamilton_nzta_traffic_count_site_point": int(len(hamilton_demo_tables["traffic_count_site_point"])),
    "hamilton_ferry_route_line": int(len(hamilton_demo_tables["ferry_route_line"])),
    "hamilton_multigeometry_road_density_map": int(len(hamilton_multigeometry_road_density_map)),
    "hamilton_place_surface_density_fact": int(len(hamilton_demo_tables["place_surface_density_fact"])),
    "hamilton_place_road_bridge": int(len(hamilton_demo_tables["place_road_bridge"])),
    "hamilton_road_surface_dimension": int(len(hamilton_demo_tables["road_surface_dimension"])),
    "hamilton_sa2_reference": int(len(hamilton_demo_tables["sa2_reference"])),
}
manifest_generated_at = datetime.now(timezone.utc).replace(microsecond=0).isoformat()

manifest = {
    "generated_at_utc": manifest_generated_at,
    "source_mode": SOURCE_MODE,
    "public_source_mode": PUBLIC_SOURCE_MODE,
    "refresh_from_source": REFRESH_FROM_SOURCE,
    "refresh_public_demo_sources": REFRESH_PUBLIC_DEMO_SOURCES,
    "artifact_source": {
        "environment_variable": "PBI_DECKGL_ARTIFACT_DIR",
        "recognized_raw_artifact_files": RAW_ARTIFACT_FILES,
        "recognized_legacy_export_files": LEGACY_EXPORT_FILES,
        "recognized_public_artifact_files": PUBLIC_ARTIFACT_FILES,
        "recognized_optional_public_artifact_files": OPTIONAL_PUBLIC_ARTIFACT_FILES,
    },
    "source_layers": SOURCE_LAYERS,
    "geometry": {
        "analysis_crs": ANALYSIS_CRS,
        "output_crs": OUTPUT_CRS,
        "wkp_precision": WKP_PRECISION,
        "export_wkt_debug": EXPORT_WKT_DEBUG,
        "place_simplify_tolerance_m": PLACE_SIMPLIFY_TOLERANCE_M,
        "road_simplify_tolerance_m": ROAD_SIMPLIFY_TOLERANCE_M,
        "legacy_geometry_note": place_road_diagnostics.get("display_simplification_note"),
        "ferry_line_geometry_note": "Ferry route lines are exported as straight A-to-B endpoints from first and last ordered GTFS shape points.",
    },
    "power_bi": {
        "visual_row_window": POWER_BI_ROW_WINDOW,
        "recommended_filters": {
            "nz_road_path": f"road_length_rank <= {POWER_BI_ROW_WINDOW}",
            "nz_sa2_travel_to_work_od_2023_arc": f"arc_is_valid = TRUE and people_count_rank <= {POWER_BI_ROW_WINDOW}",
            "nz_ferry_route_line": "line_is_valid = TRUE",
            "nz_multigeometry_road_density_map": "already row-window limited; bind as one multi-layer visual",
        },
        "field_mappings": visual_field_mapping_records(POWER_BI_ROW_WINDOW),
        "relationships": [
            "nz_place_polygon[place_feature_id] 1:* nz_place_surface_density_fact[place_feature_id]",
            "nz_road_surface_dimension[road_surface] 1:* nz_place_surface_density_fact[road_surface]",
            "nz_place_polygon[place_feature_id] 1:* nz_place_road_bridge[place_feature_id]",
            "nz_road_path[road_feature_id] 1:* nz_place_road_bridge[road_feature_id]",
            "SA2 Origin[origin_sa2_code] 1:* nz_sa2_travel_to_work_od_2023_arc[origin_sa2_code]",
            "SA2 Destination[destination_sa2_code] 1:* nz_sa2_travel_to_work_od_2023_arc[destination_sa2_code]",
        ],
    },
    "row_counts": {
        "nz_place_polygon": int(len(place_polygon)),
        "nz_road_path": int(len(road_path)),
        "nz_sa2_travel_to_work_od_2023_arc": int(len(od_arc)),
        "nz_sa2_travel_to_work_od_2023_arc_valid": valid_arc_rows,
        "nzta_traffic_count_site_point": int(len(traffic_count_site_point)),
        "nz_ferry_route_line": int(len(ferry_route_line)),
        "nz_ferry_route_line_valid": valid_ferry_line_rows,
        "nz_multigeometry_road_density_map": int(len(nz_multigeometry_road_density_map)),
        "nz_place_surface_density_fact": int(len(place_surface_density_fact)),
        "nz_place_road_bridge": int(len(place_road_bridge)),
        "nz_road_surface_dimension": int(len(road_surface_dimension)),
        "nz_sa2_reference": int(len(sa2_reference)),
    },
    "output_files": output_file_stats,
    "row_window_warnings": row_window_warnings,
    "validation": {"status": "passed" if not validation_issues else "failed", "issues": validation_issues},
    "diagnostics": diagnostics,
    "demo_datasets": {
        "hamilton_tla": {
            "output_directory": "data/power_bi/hamilton_tla_demo",
            "row_counts": hamilton_row_counts,
            "diagnostics": hamilton_demo_diagnostics,
        }
    },
}

hamilton_manifest = {
    "generated_at_utc": manifest_generated_at,
    "demo_area": "Hamilton City",
    "source_mode": SOURCE_MODE,
    "public_source_mode": PUBLIC_SOURCE_MODE,
    "source_note": "Small demo tables are derived from the generated full-NZ CSV tables and public ferry line sample.",
    "geometry": {
        "output_crs": OUTPUT_CRS,
        "wkp_precision": WKP_PRECISION,
        "ferry_line_geometry_note": "Ferry route lines are exported as straight A-to-B endpoints from first and last ordered GTFS shape points.",
    },
    "power_bi": {
        "visual_row_window": POWER_BI_ROW_WINDOW,
        "field_mappings": visual_field_mapping_records(POWER_BI_ROW_WINDOW),
        "recommended_filters": {
            "hamilton_ferry_route_line": "line_is_valid = TRUE",
            "hamilton_multigeometry_road_density_map": "none; current Hamilton rows fit under the visual window",
        },
    },
    "row_counts": hamilton_row_counts,
    "output_files": hamilton_output_file_stats,
    "validation": {
        "status": "passed" if not hamilton_validation_issues else "failed",
        "issues": hamilton_validation_issues,
    },
    "diagnostics": hamilton_demo_diagnostics,
}

manifest_path = POWER_BI_OUTPUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
hamilton_manifest_path = HAMILTON_DEMO_OUTPUT_DIR / "manifest.json"
hamilton_manifest_path.write_text(json.dumps(hamilton_manifest, indent=2), encoding="utf-8")

print("Wrote Power BI CSV outputs:")
for file_info in output_file_stats:
    print(f"  {file_info['filename']}: {file_info['rows']:,} rows, {file_info['size_bytes'] / 1_000_000:.2f} MB")
print(f"Manifest: {manifest_path}")
print("Wrote Hamilton TLA demo CSV outputs:")
for file_info in hamilton_output_file_stats:
    print(f"  {file_info['filename']}: {file_info['rows']:,} rows, {file_info['size_bytes'] / 1_000_000:.2f} MB")
print(f"Hamilton demo manifest: {hamilton_manifest_path}")
if row_window_warnings:
    print("Row window warnings:")
    for warning in row_window_warnings:
        print(f"  - {warning}")
if validation_issues:
    print("Validation failed:")
    for issue in validation_issues:
        print(f"  - {issue}")
    raise AssertionError("Export validation failed. See validation issues above.")
print("Validation passed.")


## Power BI Field Mapping

Use these bindings with the custom visual. Keep one geometry table per visual while learning the sample; once each layer works, you can combine tables with matching role columns if you want multi-layer visuals.


In [ ]:
field_mapping = pd.DataFrame(visual_field_mapping_records(POWER_BI_ROW_WINDOW))
field_mapping